# 05 · Local kinetic energy for VMC / FermiNet

Variational Monte Carlo and FermiNet need the **Laplacian** of a
log-wavefunction at every sample to evaluate the local kinetic energy
$-\tfrac12 \nabla^2\psi/\psi$. Computing it with a full autodiff Hessian costs
$O(D^2)$; omnibias gives the Laplacian of a one-layer field in **closed form**.

This notebook (JAX, float64):

1. checks the closed-form Laplacian equals `trace(jax.hessian f)`, and
2. shows it scales better as the input dimension `D` grows.

`omnibias-ferminet` wraps exactly this kernel as a drop-in
`laplacian_method` for FermiNet.

In [ ]:
import os, sys, time
os.environ["JAX_ENABLE_X64"] = "true"
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, ACCENT, GOOD, PRIMARY
set_style()

from omnibias.jax import get_activation, neural_field_laplacian, neural_field_value_grad_hessian


def make_field(key, H, D, name="tanh"):
    kW, kb, kc = jax.random.split(key, 3)
    W = jax.random.normal(kW, (H, D))
    beta = jax.random.normal(kb, (H,))
    c = jax.random.normal(kc, (H,))
    sig = get_activation(name).forward
    def f(x):
        return jnp.sum(c * sig(W @ x + beta))
    return W, beta, c, f

print("jax", jax.__version__, "x64", jax.config.read("jax_enable_x64"))

## 1. Closed-form Laplacian = trace(Hessian)

In [ ]:
H, D = 8, 5
W, beta, c, f = make_field(jax.random.PRNGKey(0), H, D)
x = jax.random.normal(jax.random.PRNGKey(1), (D,))

lap_closed = neural_field_laplacian(x, W, beta, c, "tanh")
lap_autodiff = jnp.trace(jax.hessian(f)(x))
print(f"closed-form Laplacian : {float(lap_closed): .10f}")
print(f"trace(jax.hessian f)  : {float(lap_autodiff): .10f}")
print(f"abs difference        : {abs(float(lap_closed - lap_autodiff)):.2e}")

# Local kinetic energy operator value (up to the |grad|^2 amplitude term):
print(f"kinetic term -0.5*Laplacian = {float(-0.5 * lap_closed): .6f}")

## 2. Scaling in the input dimension

Both routes are JIT-compiled; we time a single Laplacian evaluation as `D`
grows. The closed form avoids materialising the full `D×D` Hessian.

In [ ]:
def bench(fn, x, reps=50):
    fn(x).block_until_ready()  # warmup / compile
    t0 = time.perf_counter()
    for _ in range(reps):
        fn(x).block_until_ready()
    return (time.perf_counter() - t0) / reps * 1e6  # microseconds

dims = [2, 4, 8, 16, 32, 64]
t_closed, t_auto = [], []
for D in dims:
    W, beta, c, f = make_field(jax.random.PRNGKey(D), 16, D)
    x = jax.random.normal(jax.random.PRNGKey(99), (D,))
    closed = jax.jit(lambda x, W=W, beta=beta, c=c: neural_field_laplacian(x, W, beta, c, "tanh"))
    auto = jax.jit(lambda x, f=f: jnp.trace(jax.hessian(f)(x)))
    t_closed.append(bench(closed, x))
    t_auto.append(bench(auto, x))

fig, ax = plt.subplots()
ax.plot(dims, t_auto, "o-", color=ACCENT, label="trace(jax.hessian)")
ax.plot(dims, t_closed, "s-", color=GOOD, label="omnibias closed form")
ax.set_xlabel("input dimension  D"); ax.set_ylabel("time per Laplacian (µs)")
ax.set_title("Laplacian cost vs dimension (JAX, jit, CPU)")
ax.legend(); plt.show()

## Takeaway

The closed-form Laplacian matches autodiff to float64 round-off and avoids the
$O(D^2)$ Hessian, which is exactly the kinetic-energy bottleneck in VMC /
FermiNet. The `omnibias-ferminet` bridge exposes this as a drop-in
`laplacian_method` (the full-scale GPU numbers are in `docs/benchmarks.md`).

Next: **[06 · second-order optimization](06_second_order_optimization.ipynb)**.